# pLaplace JAX+PETSc API Demo

This notebook uses the maintained JAX+PETSc solver API directly on `MPI.COMM_SELF`, so the example stays interactive while still exercising the distributed backend code path.

Relevant docs:
- [docs/problems/pLaplace.md](../../docs/problems/pLaplace.md)
- [docs/results/pLaplace.md](../../docs/results/pLaplace.md)


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_ROOT = Path.cwd()
ARTIFACT_ROOT = Path("artifacts/raw_results/notebook_runs")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
PYTHON = REPO_ROOT / ".venv" / "bin" / "python"

def dump_json(path):
    with open(path, encoding="utf-8") as handle:
        return json.load(handle)


In [2]:
from mpi4py import MPI
from src.problems.plaplace.jax_petsc.solver import run_level

result = run_level(
    mesh_level=5,
    comm=MPI.COMM_SELF,
    profile="reference",
    assembly_mode="element",
    local_hessian_mode="element",
    local_coloring=True,
    nproc_threads=1,
    quiet=True,
    save_history=False,
    save_linear_timing=False,
)
{key: result[key] for key in ("mesh_level", "dofs", "energy", "iters", "total_ksp_its", "total_time")}


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


{'mesh_level': 5,
 'dofs': 3201,
 'energy': -7.9429687193,
 'iters': 6,
 'total_ksp_its': 17,
 'total_time': 0.3895}

Equivalent canonical CLI:

```bash
mpiexec -n 1 ./.venv/bin/python -u src/problems/plaplace/jax_petsc/solve_pLaplace_dof.py           --level 5 --profile reference --assembly-mode element --local-hessian-mode element           --element-reorder-mode block_xyz --local-coloring --nproc 1 --quiet
```
